In [1]:
import pandas as pd
from transformers import AutoTokenizer
import torch
from torch.utils.data import Dataset
from transformers import AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer
from transformers import pipeline

c:\Users\saram\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# قراءة الملفات
train_df = pd.read_csv(r"cnn_dailymail\train.csv")
val_df = pd.read_csv(r"cnn_dailymail\validation.csv")
test_df = pd.read_csv(r"cnn_dailymail\test.csv")


# التأكد من الأعمدة المطلوبة
print("Train columns:", train_df.columns)
print("Validation columns:", val_df.columns)
print("Test columns:", test_df.columns)

# إزالة أي صفوف ناقصة في الأعمدة الأساسية
essential_cols = ["article", "highlights"]
train_df = train_df.dropna(subset=essential_cols)
val_df = val_df.dropna(subset=essential_cols)
test_df = test_df.dropna(subset=essential_cols)


Train columns: Index(['id', 'article', 'highlights'], dtype='object')
Validation columns: Index(['id', 'article', 'highlights'], dtype='object')
Test columns: Index(['id', 'article', 'highlights'], dtype='object')


In [ ]:
model_name = "t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)

max_input_length = 512
max_target_length = 128

def preprocess(df):
    inputs = ["summarize: " + doc for doc in df["article"].tolist()]
    targets = df["highlights"].tolist()
    
    model_inputs = tokenizer(inputs, max_length=max_input_length, truncation=True, padding="max_length")
    
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(targets, max_length=max_target_length, truncation=True, padding="max_length")
    
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_dataset = preprocess(train_df)
val_dataset = preprocess(val_df)
test_dataset = preprocess(test_df)


In [ ]:
class SummarizationDataset(Dataset):
    def __init__(self, encodings):
        self.encodings = encodings

    def __len__(self):
        return len(self.encodings["input_ids"])

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        return item

train_dataset = SummarizationDataset(train_dataset)
val_dataset = SummarizationDataset(val_dataset)
test_dataset = SummarizationDataset(test_dataset)


In [ ]:
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

training_args = Seq2SeqTrainingArguments(
    output_dir="./t5-news-summarization",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    predict_with_generate=True,
    evaluation_strategy="epoch",
    logging_strategy="steps",
    logging_steps=500,
    save_strategy="epoch",
    num_train_epochs=3,
    save_total_limit=2,
    learning_rate=2e-5
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer
)

trainer.train()


In [ ]:
summarizer = pipeline("summarization", model=model, tokenizer=tokenizer)

for i in range(3):  # عرض أول 3 مقالات كمثال
    summary = summarizer(test_df["article"][i], max_length=150, min_length=40, do_sample=False)
    print("Original:", test_df["article"][i][:300], "...")
    print("Predicted Summary:", summary[0]['summary_text'])
    print("="*50)
